In [ ]:
import os, sys, platform, warnings
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB or platform.system() == 'Linux':
    os.environ.setdefault('MUJOCO_GL', 'egl')
else:
    os.environ.setdefault('MUJOCO_GL', 'glfw')
if os.environ['MUJOCO_GL'] in ('egl', 'osmesa'):
    os.environ.setdefault('PYOPENGL_PLATFORM', os.environ['MUJOCO_GL'])
if IN_COLAB:
    !{sys.executable} -m pip install -q uv
    !{sys.executable} -m uv pip install -q --python {sys.executable} "dm_control>=1.0.34" "shimmy[dm-control]" gymnasium torchinfo gdown imageio
warnings.filterwarnings('ignore', category=DeprecationWarning, module='pkg_resources')
warnings.filterwarnings('ignore', message='datetime.datetime.utcnow')
print('MuJoCo backend:', os.environ['MUJOCO_GL'])


In [ ]:
import gymnasium as gym
import numpy as np
from dm_control import suite
from shimmy import DmControlCompatibilityV0
import torch, torch.nn as nn, torch.nn.functional as F, functools
from torch.distributions.normal import Normal
import h5py, imageio, matplotlib.pyplot as plt
from tqdm import trange, tqdm
from typing import List
from torch.utils.data import Dataset, DataLoader
warnings.filterwarnings(action='ignore', message=r'datetime.datetime.utcnow')
print('All imports OK')


## Вспомогательные классы среды


In [ ]:
class DmControlWithRender(gym.Wrapper):
    def __init__(self, env, height=84, width=84, camera_id=0):
        super().__init__(env)
        self._height, self._width, self._camera_id = height, width, camera_id
    def render(self):
        dm_env = self.unwrapped._env
        return dm_env.physics.render(
            height=self._height, width=self._width, camera_id=self._camera_id)


def make_env(domain, task, frame_stack=1, flatten_frames=True,
             render_enabled=True, record_stats=False,
             transform_reward=False, gamma=0.99, hw=64):
    def thunk():
        dm_env = suite.load(domain_name=domain, task_name=task,
                            visualize_reward=False)
        env = DmControlCompatibilityV0(dm_env)
        env = gym.wrappers.FlattenObservation(env)
        if render_enabled:
            env = DmControlWithRender(env, height=hw, width=hw)
        with warnings.catch_warnings():
            warnings.filterwarnings('ignore')
            env = gym.wrappers.DtypeObservation(env, np.float32)
        if record_stats:
            env = gym.wrappers.RecordEpisodeStatistics(env)
        env = gym.wrappers.ClipAction(env)
        if transform_reward:
            env = gym.wrappers.NormalizeReward(env, gamma=gamma)
            env = gym.wrappers.TransformReward(env, lambda r: np.clip(r, -10, 10))
        env = gym.wrappers.FrameStackObservation(env, stack_size=frame_stack)
        if flatten_frames:
            env = gym.wrappers.FlattenObservation(env)
        return env
    return thunk


In [ ]:
def update_mean_var_count_from_moments(mean, var, count, batch_mean, batch_var, batch_count):
    delta = batch_mean - mean
    tot_count = count + batch_count
    new_mean = mean + delta * batch_count / tot_count
    M2 = var * count + batch_var * batch_count + torch.square(delta) * count * batch_count / tot_count
    return new_mean, M2 / tot_count, tot_count


class RunningMeanStd(nn.Module):
    def __init__(self, shape=(), epsilon=1e-4, dtype=torch.float32):
        super().__init__()
        self.register_buffer('mean', torch.zeros(shape, dtype=dtype))
        self.register_buffer('var',  torch.zeros(shape, dtype=dtype))
        self.register_buffer('count', torch.as_tensor(epsilon, dtype=dtype))
    def update(self, x):
        bm = torch.mean(x, dim=0); bv = torch.var(x, dim=0); bc = x.shape[0]
        self.mean, self.var, self.count = update_mean_var_count_from_moments(
            self.mean, self.var, self.count, bm, bv, bc)


def layer_init(layer, std=np.sqrt(2), bias_const=0.0):
    torch.nn.init.orthogonal_(layer.weight, std)
    torch.nn.init.constant_(layer.bias, bias_const)
    return layer


class Agent(nn.Module):
    def __init__(self, envs, hidden_dim=64, obs_norm_eps=1e-8):
        super().__init__()
        input_dim = np.array(envs.single_observation_space.shape).prod()
        self.obs_rms = RunningMeanStd(input_dim)
        self.obs_norm_eps = obs_norm_eps
        self.critic = nn.Sequential(
            layer_init(nn.Linear(input_dim, hidden_dim)), nn.Tanh(),
            layer_init(nn.Linear(hidden_dim, hidden_dim)), nn.Tanh(),
            layer_init(nn.Linear(hidden_dim, 1), std=1.0),
        )
        self.actor_mean = nn.Sequential(
            layer_init(nn.Linear(input_dim, hidden_dim)), nn.Tanh(),
            layer_init(nn.Linear(hidden_dim, hidden_dim)), nn.Tanh(),
            layer_init(nn.Linear(hidden_dim, np.prod(envs.single_action_space.shape)), std=0.01),
        )
        self.actor_logstd = nn.Parameter(torch.zeros(1, np.prod(envs.single_action_space.shape)))

    def normalize_obs(self, x):
        return torch.clamp((x - self.obs_rms.mean) /
                           torch.sqrt(self.obs_rms.var + self.obs_norm_eps), -10, 10)

    def get_action_and_value(self, x, action=None, greedy=False):
        nx = self.normalize_obs(x)
        action_mean = self.actor_mean(nx)
        probs = Normal(action_mean, torch.exp(self.actor_logstd.expand_as(action_mean)))
        if action is None:
            action = action_mean.detach() if greedy else probs.sample()
        return action, probs.log_prob(action).sum(1), probs.entropy().sum(1), self.critic(nx)

    def act_numpy(self, obs, greedy=False):
        obs = torch.tensor(obs[None], device=next(self.parameters()).device)
        a = self.get_action_and_value(obs, greedy=greedy)[0].cpu()
        return np.asarray(a.squeeze())


In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/HW3'
else:
    DATA_DIR = 'HW3'

checkpoint_path = os.path.join(DATA_DIR, 'cheetah-run-expert')

if not os.path.exists(os.path.join(checkpoint_path, 'checkpoint.pt')):
    import gdown
    gdown.download_folder(
        url='https://drive.google.com/drive/folders/1KAHb73CbOvmtGLKFVOHPUjQ44tFy74Mw',
        output=DATA_DIR, quiet=True)

print(checkpoint_path, sorted(os.listdir(checkpoint_path)))


def load_expert(use_cuda=False):
    device = torch.device('cuda' if torch.cuda.is_available() and use_cuda else 'cpu')
    config = torch.load(os.path.join(checkpoint_path, 'config.pt'),
                        map_location=device, weights_only=False)
    ckpt   = torch.load(os.path.join(checkpoint_path, 'checkpoint.pt'),
                        map_location=device, weights_only=False)
    domain, task = config['domain'], config['task']
    print(f'Checkpoint: {domain}/{task}, hidden_dim={config["hidden_dim"]}')
    init_env = gym.vector.SyncVectorEnv([make_env(domain, task) for _ in range(1)])
    agent = Agent(init_env, hidden_dim=config['hidden_dim']).to(device)
    agent.load_state_dict(ckpt)
    init_env.close()
    print('Expert loaded!')
    return agent


expert = load_expert(False)
expert.eval()


## 1. Оценка эксперта (3 балла)

Оцениваем предобученного PPO-эксперта:
- **Жадная политика** (`greedy=True`): всегда выбирает mean распределения.
- **Стохастическая** (`greedy=False`): семплирует из гауссова распределения.

**IQM** (Interquartile Mean) — среднее по средним 50% оценок, устойчивое к выбросам.


In [ ]:
def iqm(scores):
    s = np.sort(np.asarray(scores, dtype=float))
    n = len(s); lo, hi = n // 4, n - n // 4
    return float(np.mean(s[lo:hi]))


def iqm_std(scores):
    s = np.sort(np.asarray(scores, dtype=float))
    n = len(s); lo, hi = n // 4, n - n // 4
    vals = s[lo:hi]
    return float(np.std(vals) / np.sqrt(max(len(vals), 1)))


In [ ]:
@torch.no_grad()
def evaluate_agent(agent, env, num_episodes=20, seed=0, greedy=False, collect_frames=False):
    returns, all_frames = [], []
    label = 'greedy' if greedy else 'stochastic'
    for ep in trange(num_episodes, desc=f'Evaluating {label}'):
        obs, _ = env.reset(seed=seed + ep)
        done, total_reward, frames = False, 0.0, []
        while not done:
            if collect_frames:
                try: frames.append(env.render())
                except Exception: pass
            action = agent.act_numpy(obs, greedy=greedy)
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward
        returns.append(total_reward); all_frames.append(frames)
    return np.array(returns), all_frames


def make_gif(frames, path, fps=30):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    imageio.mimsave(path, frames, fps=fps)
    print(f'Saved GIF: {path} ({len(frames)} frames)')


def display_gif(path):
    from IPython.display import Image, display
    display(Image(filename=path))


In [ ]:
eval_env = make_env('cheetah', 'run', frame_stack=1, hw=320, render_enabled=True)()

# Greedy
greedy_returns, greedy_frames = evaluate_agent(
    expert, eval_env, num_episodes=20, seed=0, greedy=True, collect_frames=True)
print(f'\n=== Greedy policy ===')
print(f'Mean+-Std : {np.mean(greedy_returns):.1f} +- {np.std(greedy_returns):.1f}')
print(f'IQM +-SEM : {iqm(greedy_returns):.1f} +- {iqm_std(greedy_returns):.1f}')

if greedy_frames[0]:
    gif_g = os.path.join(DATA_DIR, 'expert_greedy.gif')
    make_gif(greedy_frames[0], gif_g, fps=30)
    display_gif(gif_g)
else:
    print('Rendering unavailable (Smart App Control?). Skipping GIF.')


In [ ]:
# Stochastic
stoch_returns, stoch_frames = evaluate_agent(
    expert, eval_env, num_episodes=20, seed=0, greedy=False, collect_frames=True)
print(f'\n=== Stochastic policy ===')
print(f'Mean+-Std : {np.mean(stoch_returns):.1f} +- {np.std(stoch_returns):.1f}')
print(f'IQM +-SEM : {iqm(stoch_returns):.1f} +- {iqm_std(stoch_returns):.1f}')

if stoch_frames[0]:
    gif_s = os.path.join(DATA_DIR, 'expert_stochastic.gif')
    make_gif(stoch_frames[0], gif_s, fps=30)
    display_gif(gif_s)

# Comparison table
print(f'\n{"Policy":<20} {"Mean":>8} {"Std":>8} {"IQM":>8} {"IQM_SEM":>8}')
print('-' * 56)
print(f'{"Greedy":<20} {np.mean(greedy_returns):>8.1f} {np.std(greedy_returns):>8.1f}'
      f' {iqm(greedy_returns):>8.1f} {iqm_std(greedy_returns):>8.1f}')
print(f'{"Stochastic":<20} {np.mean(stoch_returns):>8.1f} {np.std(stoch_returns):>8.1f}'
      f' {iqm(stoch_returns):>8.1f} {iqm_std(stoch_returns):>8.1f}')
eval_env.close()


## 2. Сбор датасета (2 балла)

Собираем экспертные траектории жадной политикой и сохраняем в HDF5.
Структура: каждая группа — одна траектория, содержит `states (T,17)` и `actions (T,6)`.
Сохраняем **сырые** (не стекированные) наблюдения — стекинг делает StackedDataset.


In [ ]:
DATASET_PATH = os.path.join(DATA_DIR, 'cheetah_expert_1024.h5')


@torch.no_grad()
def collect_expert_dataset(agent, num_episodes=1024, seed=0, save_path=DATASET_PATH):
    col_env = make_env('cheetah', 'run', frame_stack=1, render_enabled=False)()
    with h5py.File(save_path, 'w') as f:
        for ep in trange(num_episodes, desc='Collecting expert data'):
            obs, _ = col_env.reset(seed=seed + ep)
            done = False
            states_ep, actions_ep = [], []
            while not done:
                action = agent.act_numpy(obs, greedy=True)
                states_ep.append(obs.copy())
                actions_ep.append(action.copy())
                obs, _, terminated, truncated, _ = col_env.step(action)
                done = terminated or truncated
            grp = f.create_group(str(ep))
            grp.create_dataset('states',  data=np.array(states_ep,  dtype=np.float32))
            grp.create_dataset('actions', data=np.array(actions_ep, dtype=np.float32))
    col_env.close()
    print(f'Saved {num_episodes} trajectories -> {save_path}')
    return save_path


if not os.path.exists(DATASET_PATH):
    collect_expert_dataset(expert, num_episodes=1024, seed=0)
else:
    print(f'Dataset already exists: {DATASET_PATH}')

# Verify
with h5py.File(DATASET_PATH, 'r') as f:
    keys = sorted(f.keys(), key=int)
    n_traj = len(keys)
    s0 = f[keys[0]]['states'].shape
    a0 = f[keys[0]]['actions'].shape
    total = sum(f[k]['states'].shape[0] for k in keys)
print(f'Trajectories : {n_traj}')
print(f'States shape : {s0}  (T, obs_dim=17)')
print(f'Actions shape: {a0} (T, act_dim=6)')
print(f'Total steps  : {total:,}  |  Mean ep len: {total/n_traj:.1f}')


## Подготовка датасета — StackedDataset


In [ ]:
def sort_keys(keys):
    return [str(k) for k in sorted(int(x) for x in keys)]


class StackedDataset(Dataset):
    def __init__(self, hdf5_path, frame_stack=1, device='cpu',
                 max_video_len=None, max_videos=None):
        with h5py.File(hdf5_path, 'r') as df:
            keys = sort_keys(df.keys())[:max_videos]
            self.states  = [torch.tensor(df[t]['states'][:max_video_len],  device=device) for t in keys]
            self.actions = [torch.tensor(df[t]['actions'][:max_video_len], device=device) for t in keys]
        self.traj_len  = self.states[0].shape[0]
        self.state_dim  = self.states[0][0].shape[-1]
        self.action_dim = self.actions[0][0].shape[-1]
        self.device      = device
        self.frame_stack = frame_stack
        print(f'Loaded {len(self.states)} trajs | state_dim={self.state_dim} | act_dim={self.action_dim}')

    def _get_padded_state(self, traj_idx, idx):
        lo = max(0, idx - self.frame_stack + 1)
        states = self.states[traj_idx][lo:idx + 1]
        k = len(states)
        if k < self.frame_stack:
            pad = [states[0][None]] * (self.frame_stack - k)
            states = torch.cat(pad + [states])
        return states.reshape(-1)

    def __len__(self):
        return len(self.actions) * self.traj_len

    def __getitem__(self, idx):
        traj_idx, trans_idx = divmod(idx, self.traj_len)
        return self._get_padded_state(traj_idx, trans_idx), self.actions[traj_idx][trans_idx]


In [ ]:
use_cuda    = torch.cuda.is_available()
frame_stack = 2
batch_size  = 256
dataset_path = DATASET_PATH
device = torch.device('cuda' if use_cuda else 'cpu')
print(f'Device: {device}')

dataset = StackedDataset(dataset_path, frame_stack, device)

dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)
print(f'Batches per epoch: {len(dataloader)}')


## 3. Архитектура Behavioral Cloning (4 балла)

### MLPBlock (2 балла)
Блок: Linear -> LayerNorm -> ReLU -> Dropout
LayerNorm стабилизирует обучение без зависимости от размера батча.

### MLPActHead (2 балла)
Стек MLPBlock + линейный выходной слой.
Вход: стекированные наблюдения `obs_dim * frame_stack = 17*2 = 34`.
Выход: вектор действий размерности `act_dim = 6` (без активации — регрессия).


In [ ]:
class MLPBlock(nn.Module):
    # Linear(in->out) -> LayerNorm(out) -> ReLU -> Dropout(p)
    def __init__(self, input_dim: int, output_dim: int, dropout: float = 0.05):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, output_dim),
            nn.LayerNorm(output_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# Quick sanity check
_b = MLPBlock(34, 256)
assert _b(torch.randn(4, 34)).shape == (4, 256), 'MLPBlock shape mismatch'
print('MLPBlock OK')


In [ ]:
class MLPActHead(nn.Module):
    # Stack of MLPBlocks + linear output head (no activation)
    def __init__(self, obs_dim: int, act_dim: int,
                 hidden_dim: int = 256, num_layers: int = 3, dropout: float = 0.05):
        super().__init__()
        layers = []
        in_dim = obs_dim
        for _ in range(num_layers):
            layers.append(MLPBlock(in_dim, hidden_dim, dropout))
            in_dim = hidden_dim
        layers.append(nn.Linear(hidden_dim, act_dim))
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


obs_dim = dataset.state_dim * dataset.frame_stack   # 17 * 2 = 34
act_dim = dataset.action_dim                         # 6

_m = MLPActHead(obs_dim, act_dim)
assert _m(torch.randn(4, obs_dim)).shape == (4, act_dim), 'MLPActHead shape mismatch'
print(f'MLPActHead OK: input={obs_dim} -> hidden=256x3 -> output={act_dim}')

bc_actor = MLPActHead(obs_dim, act_dim, hidden_dim=256, num_layers=3, dropout=0.05).to(device)
print(bc_actor)


## Обучение Behavioral Cloning (1 балл)

BC = задача регрессии: минимизируем `MSE(predicted_action, expert_action)`.


In [ ]:
def get_optim_groups(model, weight_decay):
    return [
        {'params': (p for p in model.parameters() if p.dim() < 2),  'weight_decay': 0.0},
        {'params': (p for p in model.parameters() if p.dim() >= 2), 'weight_decay': weight_decay},
    ]


def _linear_decay_warmup(iteration, warmup_iterations, total_iterations):
    if iteration < warmup_iterations:
        return iteration / warmup_iterations
    return 1.0 - (iteration - warmup_iterations) / (total_iterations - warmup_iterations)


def linear_annealing_with_warmup(optimizer, warmup_steps, total_steps):
    fn = functools.partial(_linear_decay_warmup,
                           warmup_iterations=warmup_steps, total_iterations=total_steps)
    return torch.optim.lr_scheduler.LambdaLR(optimizer, fn)


In [ ]:
num_epochs    = 15
warmup_epochs = 1

optim = torch.optim.AdamW(get_optim_groups(bc_actor, 1e-4), lr=1e-3)
total_updates  = len(dataloader) * num_epochs
warmup_updates = len(dataloader) * warmup_epochs
scheduler = linear_annealing_with_warmup(optim, warmup_updates, total_updates)

train_losses = []

for epoch in range(num_epochs):
    bc_actor.train()
    epoch_loss, n = 0.0, 0
    for states, target_action in dataloader:
        pred_action = bc_actor(states)
        loss = F.mse_loss(pred_action, target_action)
        optim.zero_grad(set_to_none=True); loss.backward()
        optim.step(); scheduler.step()
        epoch_loss += loss.item() * states.size(0); n += states.size(0)
    avg = epoch_loss / n
    train_losses.append(avg)
    print(f'Epoch {epoch+1:2d}/{num_epochs}  loss={avg:.5f}  lr={scheduler.get_last_lr()[0]:.6f}')

plt.figure(figsize=(7, 3))
plt.plot(range(1, num_epochs + 1), train_losses, marker='o')
plt.xlabel('Epoch'); plt.ylabel('MSE loss'); plt.title('BC Training Loss')
plt.tight_layout(); plt.show()


In [ ]:
@torch.no_grad()
def evaluate_actor(env, bc_actor, num_episodes=50, seed=0):
    returns = []
    bc_actor.eval()
    for ep in trange(num_episodes, desc='Evaluating BC', leave=False):
        obs, _ = env.reset(seed=seed + ep)
        done, total_reward = False, 0.0
        while not done:
            obs_t  = torch.tensor(obs.copy(), device=device)[None].contiguous()
            action = bc_actor(obs_t).squeeze().cpu().numpy()
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated; total_reward += reward
        returns.append(total_reward)
    return np.array(returns)


bc_eval_env = make_env('cheetah', 'run', frame_stack=frame_stack,
                        flatten_frames=True, render_enabled=False)()
bc_returns = evaluate_actor(bc_eval_env, bc_actor, num_episodes=50, seed=0)
bc_eval_env.close()

print(f'\nBC Agent (full dataset, {num_epochs} epochs)')
print(f'Mean+-Std : {np.mean(bc_returns):.1f} +- {np.std(bc_returns):.1f}')
print(f'IQM +-SEM : {iqm(bc_returns):.1f} +- {iqm_std(bc_returns):.1f}')
print(f'Min / Max : {bc_returns.min():.1f} / {bc_returns.max():.1f}')


## 4. Анализ зависимости BC от размера датасета (5 баллов)

Обучаем BC на разных размерах: [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024] траекторий.
Оцениваем:
- **In-distribution** (seed 0-49): те же начальные состояния, что в датасете.
- **Out-of-distribution** (seed 5000-5049): новые состояния, которых не было в датасете.

IQM устойчивее к выбросам, чем среднее, поэтому используем оба.


In [ ]:
dataset_sizes = [2, 4, 8, 16, 32, 64, 128, 256, 512, 1024]
n_eval_bc     = 50
eval_epochs   = 15

res = {k: [] for k in ['sizes','in_iqm','in_sem','in_mean','in_std',
                                  'out_iqm','out_sem','out_mean','out_std']}
res['sizes'] = dataset_sizes

_env_in  = make_env('cheetah','run',frame_stack=frame_stack,flatten_frames=True,render_enabled=False)()
_env_out = make_env('cheetah','run',frame_stack=frame_stack,flatten_frames=True,render_enabled=False)()

for n_traj in dataset_sizes:
    print(f'\n--- {n_traj} trajectories ---')
    ds = StackedDataset(dataset_path, frame_stack, device, max_videos=n_traj)
    dl = DataLoader(ds, batch_size=min(batch_size, len(ds)), shuffle=True,
                    drop_last=(len(ds) >= batch_size))

    actor = MLPActHead(obs_dim, act_dim, hidden_dim=256, num_layers=3, dropout=0.05).to(device)
    opt   = torch.optim.AdamW(get_optim_groups(actor, 1e-4), lr=1e-3)
    sch   = linear_annealing_with_warmup(opt, max(1, len(dl)), len(dl)*eval_epochs)

    actor.train()
    for _ep in range(eval_epochs):
        for s, a in dl:
            loss = F.mse_loss(actor(s), a)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step(); sch.step()

    ret_in  = evaluate_actor(_env_in,  actor, n_eval_bc, seed=0)
    ret_out = evaluate_actor(_env_out, actor, n_eval_bc, seed=5000)

    res['in_iqm'].append(iqm(ret_in));    res['in_sem'].append(iqm_std(ret_in))
    res['in_mean'].append(np.mean(ret_in)); res['in_std'].append(np.std(ret_in))
    res['out_iqm'].append(iqm(ret_out));   res['out_sem'].append(iqm_std(ret_out))
    res['out_mean'].append(np.mean(ret_out)); res['out_std'].append(np.std(ret_out))
    print(f'In : IQM={iqm(ret_in):.1f}+-{iqm_std(ret_in):.1f}  Mean={np.mean(ret_in):.1f}')
    print(f'Out: IQM={iqm(ret_out):.1f}+-{iqm_std(ret_out):.1f} Mean={np.mean(ret_out):.1f}')

_env_in.close(); _env_out.close()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
szs = res['sizes']

for ax, (ym, ys, om, os_), title in zip(
    axes,
    [('in_iqm','in_sem','out_iqm','out_sem'), ('in_mean','in_std','out_mean','out_std')],
    ['IQM +- SEM', 'Mean +- Std']):
    ym_, ys_, om_, os__ = [res[k] for k in (ym, ys, om, os_)]
    ax.errorbar(szs, ym_, ys_, marker='o', capsize=4, label='In-distribution')
    ax.errorbar(szs, om_, os__, marker='s', capsize=4, label='Out-of-distribution', ls='--')
    ax.set_xscale('log'); ax.set_xlabel('Expert trajectories')
    ax.set_ylabel('Episode return'); ax.set_title(title)
    ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('BC quality vs dataset size (cheetah-run)')
plt.tight_layout(); plt.show()


### Интерпретация

- **Малый датасет (2-8)**: высокое in-distribution качество может быть случайностью или переобучением;
  out-of-distribution значительно хуже — BC не обобщает.
- **Рост датасета**: обе кривые растут и сближаются — covariate shift снижается.
- **Разрыв in/out** всегда присутствует: BC видела только состояния эксперта.
  При малейшем отклонении политики она попадает в незнакомые состояния.


## 5. DAgger — Dataset Aggregation (5 баллов)

**Covariate shift**: BC обучена на состояниях эксперта, но реально попадает в другие.
Маленькие ошибки накапливаются — политика уходит всё дальше от обученного распределения.

**DAgger** решает это итерационно:
1. Текущая BC-политика собирает траектории (исследует свои состояния).
2. Эксперт **размечает** эти состояния оптимальными действиями.
3. Размеченные данные добавляются в датасет.
4. BC переобучается с нуля на накопленном датасете.

Со временем датасет покрывает состояния самой BC-политики -> covariate shift исчезает.


In [ ]:
class UpdateDataset(StackedDataset):
    # StackedDataset with update() to add new trajectories after each DAgger iteration
    def update(self, new_states, new_actions):
        assert new_states[0].shape[-1] == self.state_dim, (
            f'Expected raw states of dim {self.state_dim}, got {new_states[0].shape[-1]}!')
        self.states.extend([
            torch.tensor(s, device=self.device, dtype=torch.float32) for s in new_states])
        self.actions.extend([
            torch.tensor(a, device=self.device, dtype=torch.float32) for a in new_actions])


In [ ]:
@torch.no_grad()
def log_actor_trajectories_with_expert_actions(
    env, expert: Agent, bc_actor: MLPActHead,
    num_episodes: int = 1, seed: int = 0,
) -> tuple:
    # BC navigates; expert labels each state. Returns raw (non-stacked) states.
    raw_dim = dataset.state_dim   # 17 for cheetah-run
    states_arr, actions_arr, returns = [], [], []

    for ep in trange(num_episodes, desc='DAgger collect', leave=False):
        states, actions = [], []
        total_reward = 0.0
        obs, info = env.reset(seed=seed + ep)
        done = False

        while not done:
            # BC actor selects action using stacked obs (34-dim)
            obs_t     = torch.tensor(obs.copy(), device=device)[None].contiguous()
            bc_action = bc_actor(obs_t).squeeze().cpu().numpy()

            # Expert labels the CURRENT raw state
            # With frame_stack=2: obs = [obs_{t-1}, obs_t]; take obs_t = last raw_dim elements
            raw_obs       = obs[-raw_dim:].copy()
            expert_action = expert.act_numpy(raw_obs, greedy=True)

            states.append(raw_obs)
            actions.append(expert_action)

            obs, reward, terminated, truncated, info = env.step(bc_action)
            done = terminated or truncated
            total_reward += reward

        returns.append(total_reward)
        states_arr.append(np.array(states,  dtype=np.float32))
        actions_arr.append(np.array(actions, dtype=np.float32))

    return states_arr, actions_arr, np.array(returns)


In [ ]:
N_DAGGER_ITER = 10
N_COLLECT     = 10     # episodes per iteration
INIT_TRAJ     = 100    # start with first 100 expert trajectories

dagger_env = make_env('cheetah', 'run', frame_stack=frame_stack,
                       flatten_frames=True, render_enabled=False)()

dagger_dataset = UpdateDataset(dataset_path, frame_stack=frame_stack, device=device,
                                max_videos=INIT_TRAJ)
dagger_dl = DataLoader(dagger_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

dag_res = {'iter':[], 'iqm':[], 'iqm_std':[], 'mean':[], 'std':[]}

for iteration in range(N_DAGGER_ITER):
    print(f'\nDAgger iter {iteration} | dataset: {len(dagger_dataset)} transitions')

    # 1. Train fresh BC from scratch
    bc_actor_dag = MLPActHead(obs_dim, act_dim, hidden_dim=256,
                               num_layers=3, dropout=0.05).to(device)
    opt_d = torch.optim.AdamW(get_optim_groups(bc_actor_dag, 1e-4), lr=1e-3)
    sch_d = linear_annealing_with_warmup(opt_d, max(1, len(dagger_dl)),
                                          len(dagger_dl) * num_epochs)
    bc_actor_dag.train()
    for _ in range(num_epochs):
        for s, a in dagger_dl:
            loss = F.mse_loss(bc_actor_dag(s), a)
            opt_d.zero_grad(set_to_none=True); loss.backward(); opt_d.step(); sch_d.step()

    # 2. Collect trajectories with BC; expert labels
    new_states, new_actions, ep_returns = log_actor_trajectories_with_expert_actions(
        dagger_env, expert, bc_actor_dag,
        num_episodes=N_COLLECT, seed=5000 + iteration * N_COLLECT)

    # 3. Update dataset
    dagger_dataset.update(new_states, new_actions)
    dagger_dl = DataLoader(dagger_dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    # 4. Log metrics
    dag_res['iter'].append(iteration)
    dag_res['iqm'].append(iqm(ep_returns));      dag_res['iqm_std'].append(iqm_std(ep_returns))
    dag_res['mean'].append(np.mean(ep_returns)); dag_res['std'].append(np.std(ep_returns))
    print(f'IQM={iqm(ep_returns):.1f}+-{iqm_std(ep_returns):.1f}  '
          f'Mean={np.mean(ep_returns):.1f}+-{np.std(ep_returns):.1f}')

bc_actor = bc_actor_dag   # keep final policy
dagger_env.close()


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.errorbar(dag_res['iter'], dag_res['iqm'], dag_res['iqm_std'],
             marker='o', capsize=4, color='steelblue')
ax1.set_xlabel('DAgger iteration'); ax1.set_ylabel('IQM return')
ax1.set_title('DAgger: IQM per iteration'); ax1.grid(True, alpha=0.3)

ax2.errorbar(dag_res['iter'], dag_res['mean'], dag_res['std'],
             marker='s', capsize=4, color='darkorange')
ax2.set_xlabel('DAgger iteration'); ax2.set_ylabel('Mean return')
ax2.set_title('DAgger: Mean +- Std per iteration'); ax2.grid(True, alpha=0.3)

plt.suptitle('DAgger convergence (cheetah-run)')
plt.tight_layout(); plt.show()


### Сравнение BC и DAgger

| Метод | Данные | Covariate Shift |
|-------|--------|------------------|
| **BC** | Только экспертные траектории | Присутствует: политика уходит от обученного распределения |
| **DAgger** | Эксперт + собственные траектории BC | Снижается итерационно |

DAgger использует **сопоставимое количество экспертных меток**, но распределяет их по состояниям,
которые реально посещает текущая политика. Это устраняет накопление ошибок,
и качество BC монотонно растёт с каждой итерацией, тогда как обычный BC 'застывает'.
